In [41]:
import sys
sys.path.append("..")  # so we can import from src/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn as sk
from sklearn.model_selection import StratifiedGroupKFold

from src.data_loader import load_raw_data, load_id_mappings, add_target

In [35]:
df = load_raw_data()
df.set_index('encounter_id', inplace=True)
df.shape

(101766, 49)

In [36]:
#removing expired and hospice patients

exp_hsp_pat = [11, 13, 14, 19, 20]
df = df[~df['discharge_disposition_id'].isin(exp_hsp_pat)]
df.shape

(99343, 49)

In [37]:
#dropping weight as it has 98569 rows as blank
df.drop('weight', axis=1, inplace=True)
df.shape

(99343, 48)

In [38]:
#filling missing values with unknown
cols = ["race", "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3"]
df[cols] = df[cols].fillna("Unknown")

In [39]:
#converting age map into numeric
age_map = {"[0-10)":1,"[10-20)":2,"[20-30)":3,"[30-40)":4,"[40-50)":5,"[50-60)":6,"[60-70)":7,"[70-80)":8,"[80-90)":9,"[90-100)":10}
df['age'] = df['age'].map(age_map)
#df.head()
#df['age'].isna().sum()


In [40]:
#creating the target column
Y = (df['readmitted'] == '<30').astype(int)
print(Y)
df.drop('readmitted', axis=1, inplace=True)
X = df
X.shape

encounter_id
2278392      0
149190       0
64410        0
500364       0
16680        0
            ..
443847548    0
443847782    0
443854148    0
443857166    0
443867222    0
Name: readmitted, Length: 99343, dtype: int64


(99343, 47)

In [42]:
#split the data by patient number


<bound method NDFrame.head of               patient_nbr             race  gender  age  admission_type_id  \
encounter_id                                                                 
2278392           8222157        Caucasian  Female    1                  6   
149190           55629189        Caucasian  Female    2                  1   
64410            86047875  AfricanAmerican  Female    3                  1   
500364           82442376        Caucasian    Male    4                  1   
16680            42519267        Caucasian    Male    5                  1   
...                   ...              ...     ...  ...                ...   
443847548       100162476  AfricanAmerican    Male    8                  1   
443847782        74694222  AfricanAmerican  Female    9                  1   
443854148        41088789        Caucasian    Male    8                  1   
443857166        31693671        Caucasian  Female    9                  2   
443867222       175429310        C